# 🔧 AI4I 2020 Predictive Maintenance
### End-to-End ML Pipeline: EDA → Feature Engineering → Model Training → FastAPI Export
**Models:** Random Forest | XGBoost  
**Dataset:** AI4I 2020 Predictive Maintenance Dataset (UCI)

## 📦 Step 1: Install & Import Libraries

In [ ]:
# Install required libraries (run once)
!pip install xgboost scikit-learn pandas numpy matplotlib seaborn joblib imbalanced-learn --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve,
    f1_score, accuracy_score
)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('✅ All libraries imported successfully!')

## 📂 Step 2: Load Dataset

In [ ]:
# Load the dataset — update path if needed
df = pd.read_csv('ai4i2020.csv')

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
# Basic info
df.info()

In [ ]:
# Statistical summary
df.describe().round(2)

## 🔍 Step 3: Exploratory Data Analysis (EDA)

In [ ]:
# ── 3.1 Missing values & duplicates ──────────────────────
print('Missing values:')
print(df.isnull().sum())
print(f'\nDuplicate rows: {df.duplicated().sum()}')

In [ ]:
# ── 3.2 Target class distribution ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Machine failure count
failure_counts = df['Machine failure'].value_counts()
axes[0].bar(['No Failure (0)', 'Failure (1)'], failure_counts.values,
            color=['#2ecc71', '#e74c3c'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Machine Failure Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(failure_counts.values):
    axes[0].text(i, v + 30, f'{v}\n({v/len(df)*100:.1f}%)', ha='center', fontweight='bold')

# Machine type distribution
type_counts = df['Type'].value_counts()
axes[1].pie(type_counts.values, labels=type_counts.index,
            autopct='%1.1f%%', colors=['#3498db', '#e67e22', '#9b59b6'],
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Machine Type Distribution', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('eda_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nClass imbalance ratio: 1:{int(failure_counts[0]/failure_counts[1])} (No Failure : Failure)')

In [ ]:
# ── 3.3 Failure type breakdown ────────────────────────────
failure_cols = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
failure_labels = [
    'Tool Wear\nFailure', 'Heat\nDissipation', 'Power\nFailure',
    'Overstrain\nFailure', 'Random\nFailure'
]

fig, ax = plt.subplots(figsize=(10, 4))
counts = [df[c].sum() for c in failure_cols]
bars = ax.bar(failure_labels, counts,
              color=['#e74c3c', '#f39c12', '#3498db', '#9b59b6', '#1abc9c'],
              edgecolor='white', linewidth=1.5)
ax.set_title('Failure Type Distribution', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Failures')
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(count), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('eda_failure_types.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 3.4 Feature distributions by failure label ────────────
numeric_cols = [
    'Air temperature [K]', 'Process temperature [K]',
    'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]'
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    for label, color, alpha in [(0, '#2ecc71', 0.6), (1, '#e74c3c', 0.8)]:
        axes[i].hist(
            df[df['Machine failure'] == label][col],
            bins=40, color=color, alpha=alpha,
            label='Failure' if label else 'No Failure', density=True
        )
    axes[i].set_title(col, fontsize=10, fontweight='bold')
    axes[i].legend(fontsize=8)
    axes[i].set_ylabel('Density')

axes[5].axis('off')  # hide empty subplot
fig.suptitle('Feature Distributions: Failure vs No Failure', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('eda_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 3.5 Correlation heatmap ───────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))
corr_cols = numeric_cols + ['Machine failure']
corr = df[corr_cols].corr()

mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, ax=ax, square=True,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 3.6 Failure rate by machine type ─────────────────────
failure_by_type = df.groupby('Type')['Machine failure'].agg(['sum', 'count'])
failure_by_type['failure_rate'] = (failure_by_type['sum'] / failure_by_type['count'] * 100).round(2)
print('Failure rate by machine type:')
print(failure_by_type)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(failure_by_type.index, failure_by_type['failure_rate'],
       color=['#3498db', '#e67e22', '#9b59b6'], edgecolor='white', linewidth=1.5)
ax.set_title('Failure Rate by Machine Type (%)', fontsize=12, fontweight='bold')
ax.set_ylabel('Failure Rate (%)')
for i, (idx, row) in enumerate(failure_by_type.iterrows()):
    ax.text(i, row['failure_rate'] + 0.05, f"{row['failure_rate']}%", ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('eda_failure_by_type.png', dpi=150, bbox_inches='tight')
plt.show()

## ⚙️ Step 4: Feature Engineering

In [ ]:
df_eng = df.copy()

# ── 4.1 Temperature delta ─────────────────────────────────
df_eng['temp_delta'] = df_eng['Process temperature [K]'] - df_eng['Air temperature [K]']

# ── 4.2 Power = Torque × Angular velocity ─────────────────
# ω (rad/s) = RPM × 2π / 60
df_eng['power_watts'] = df_eng['Torque [Nm]'] * (df_eng['Rotational speed [rpm]'] * 2 * np.pi / 60)

# ── 4.3 Tool wear rate (torque × wear) ────────────────────
df_eng['wear_torque_interaction'] = df_eng['Tool wear [min]'] * df_eng['Torque [Nm]']

# ── 4.4 Speed-torque ratio ────────────────────────────────
df_eng['speed_torque_ratio'] = df_eng['Rotational speed [rpm]'] / (df_eng['Torque [Nm]'] + 1e-6)

# ── 4.5 Tool wear bins (categorical) ─────────────────────
df_eng['wear_bin'] = pd.cut(
    df_eng['Tool wear [min]'],
    bins=[0, 60, 120, 180, 300],
    labels=[0, 1, 2, 3],  # Low / Medium / High / Critical
    include_lowest=True   # ensures Tool wear == 0 is captured
).astype('Int64').fillna(0).astype(int)

# ── 4.6 High temp flag ────────────────────────────────────
df_eng['high_temp_flag'] = (df_eng['Process temperature [K]'] > df_eng['Process temperature [K]'].quantile(0.90)).astype(int)

# ── 4.7 Low speed flag ────────────────────────────────────
df_eng['low_speed_flag'] = (df_eng['Rotational speed [rpm]'] < df_eng['Rotational speed [rpm]'].quantile(0.10)).astype(int)

print('✅ Engineered features added:')
new_feats = ['temp_delta', 'power_watts', 'wear_torque_interaction',
             'speed_torque_ratio', 'wear_bin', 'high_temp_flag', 'low_speed_flag']
print(df_eng[new_feats].describe().round(2))

## 🧹 Step 5: Preprocessing

In [ ]:
# ── 5.1 Encode machine type ───────────────────────────────
le = LabelEncoder()
df_eng['Type_encoded'] = le.fit_transform(df_eng['Type'])
print('Type encoding:', dict(zip(le.classes_, le.transform(le.classes_))))

# ── 5.2 Define feature set ────────────────────────────────
FEATURE_COLS = [
    'Type_encoded',
    'Air temperature [K]',
    'Process temperature [K]',
    'Rotational speed [rpm]',
    'Torque [Nm]',
    'Tool wear [min]',
    # Engineered features
    'temp_delta',
    'power_watts',
    'wear_torque_interaction',
    'speed_torque_ratio',
    'wear_bin',
    'high_temp_flag',
    'low_speed_flag'
]
TARGET_COL = 'Machine failure'

X = df_eng[FEATURE_COLS]
y = df_eng[TARGET_COL]

print(f'\nFeatures shape: {X.shape}')
print(f'Target distribution:\n{y.value_counts()}')

In [ ]:
# ── 5.3 Train/test split ──────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ── 5.4 Scale features ────────────────────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# ── 5.5 SMOTE to handle class imbalance ──────────────────
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print(f'Before SMOTE: {pd.Series(y_train).value_counts().to_dict()}')
print(f'After  SMOTE: {pd.Series(y_train_resampled).value_counts().to_dict()}')

## 🌲 Step 6: Train Random Forest

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_resampled, y_train_resampled)

rf_pred  = rf_model.predict(X_test_scaled)
rf_proba = rf_model.predict_proba(X_test_scaled)[:, 1]

print('=== Random Forest Results ===')
print(classification_report(y_test, rf_pred, target_names=['No Failure', 'Failure']))
print(f'ROC-AUC Score: {roc_auc_score(y_test, rf_proba):.4f}')

## ⚡ Step 7: Train XGBoost

In [ ]:
# Calculate scale_pos_weight for XGBoost (alternative to SMOTE)
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count
print(f'scale_pos_weight: {scale_pos_weight:.2f}')

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train_resampled, y_train_resampled,
    eval_set=[(X_test_scaled, y_test)],
    verbose=False
)

xgb_pred  = xgb_model.predict(X_test_scaled)
xgb_proba = xgb_model.predict_proba(X_test_scaled)[:, 1]

print('\n=== XGBoost Results ===')
print(classification_report(y_test, xgb_pred, target_names=['No Failure', 'Failure']))
print(f'ROC-AUC Score: {roc_auc_score(y_test, xgb_proba):.4f}')

## 💾 Step 7b: Save Model Artifacts for FastAPI

In [ ]:
import os, json, joblib

# ── Save all model artifacts for FastAPI ──────────────────
os.makedirs('model/artifacts', exist_ok=True)

# Individual models
joblib.dump(rf_model,     'model/artifacts/rf_model.pkl')
joblib.dump(xgb_model,    'model/artifacts/xgb_model.pkl')
joblib.dump(scaler,       'model/artifacts/scaler.pkl')
joblib.dump(le,           'model/artifacts/label_encoder.pkl')
joblib.dump(FEATURE_COLS, 'model/artifacts/feature_cols.pkl')

# Best model by ROC-AUC (evaluated on test set)
rf_auc_save  = roc_auc_score(y_test, rf_model.predict_proba(X_test_scaled)[:, 1])
xgb_auc_save = roc_auc_score(y_test, xgb_model.predict_proba(X_test_scaled)[:, 1])

if xgb_auc_save >= rf_auc_save:
    best_model_obj  = xgb_model
    best_model_name = 'XGBoost'
    best_auc_val    = xgb_auc_save
else:
    best_model_obj  = rf_model
    best_model_name = 'RandomForest'
    best_auc_val    = rf_auc_save

joblib.dump(best_model_obj, 'model/artifacts/model.pkl')

# Save metadata
metadata = {
    'best_model':    best_model_name,
    'best_roc_auc':  round(best_auc_val, 4),
    'rf_roc_auc':    round(rf_auc_save, 4),
    'xgb_roc_auc':   round(xgb_auc_save, 4),
    'feature_cols':  FEATURE_COLS,
    'label_classes': list(le.classes_),
    'smote_used':    True
}
with open('model/artifacts/metadata.json', 'w') as mf:
    json.dump(metadata, mf, indent=2)

# Verify all files saved correctly
print('✅ Artifacts saved to model/artifacts/:')
for fname in sorted(os.listdir('model/artifacts')):
    size = os.path.getsize(f'model/artifacts/{fname}')
    print(f'   📦 {fname:35s}  {size/1024:.1f} KB')

print(f'\n🏆 Best model : {best_model_name}')
print(f'   RF  AUC    : {rf_auc_save:.4f}')
print(f'   XGB AUC    : {xgb_auc_save:.4f}')

## 📊 Step 8: Model Comparison & Evaluation

In [ ]:
# ── 8.1 Metrics summary table ─────────────────────────────
from sklearn.metrics import precision_score, recall_score

results = pd.DataFrame({
    'Model': ['Random Forest', 'XGBoost'],
    'Accuracy':  [accuracy_score(y_test, rf_pred),  accuracy_score(y_test, xgb_pred)],
    'Precision': [precision_score(y_test, rf_pred), precision_score(y_test, xgb_pred)],
    'Recall':    [recall_score(y_test, rf_pred),    recall_score(y_test, xgb_pred)],
    'F1-Score':  [f1_score(y_test, rf_pred),        f1_score(y_test, xgb_pred)],
    'ROC-AUC':   [roc_auc_score(y_test, rf_proba),  roc_auc_score(y_test, xgb_proba)]
}).round(4)

results.set_index('Model', inplace=True)
print(results.to_string())

In [ ]:
# ── 8.2 Confusion matrices ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
titles = ['Random Forest', 'XGBoost']
preds  = [rf_pred, xgb_pred]

for ax, title, pred in zip(axes, titles, preds):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No Failure', 'Failure'],
                yticklabels=['No Failure', 'Failure'],
                linewidths=0.5, cbar=False)
    ax.set_title(f'{title}\nConfusion Matrix', fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig('eval_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 8.3 ROC curves ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, title, proba in zip(axes, titles, [rf_proba, xgb_proba]):
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, color='#e74c3c', lw=2, label=f'ROC (AUC = {auc:.3f})')
    ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
    ax.fill_between(fpr, tpr, alpha=0.1, color='#e74c3c')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'{title} — ROC Curve', fontsize=12, fontweight='bold')
    ax.legend()

plt.tight_layout()
plt.savefig('eval_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 8.4 Feature importance comparison ────────────────────
rf_importances  = pd.Series(rf_model.feature_importances_,  index=FEATURE_COLS)
xgb_importances = pd.Series(xgb_model.feature_importances_, index=FEATURE_COLS)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, imp, title, color in zip(
    axes,
    [rf_importances, xgb_importances],
    ['Random Forest', 'XGBoost'],
    ['#27ae60', '#2980b9']
):
    imp_sorted = imp.sort_values(ascending=True)
    ax.barh(imp_sorted.index, imp_sorted.values, color=color, alpha=0.85)
    ax.set_title(f'{title}\nFeature Importances', fontsize=12, fontweight='bold')
    ax.set_xlabel('Importance Score')

plt.tight_layout()
plt.savefig('eval_feature_importances.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 8.5 Cross-validation ──────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in [('Random Forest', rf_model), ('XGBoost', xgb_model)]:
    cv_scores = cross_val_score(model, X_train_resampled, y_train_resampled,
                                cv=cv, scoring='roc_auc', n_jobs=1)  # avoid parallelism conflict with XGBoost threading
    print(f'{name} — CV ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

## 🏆 Step 9: Select Best Model

In [ ]:
rf_auc  = roc_auc_score(y_test, rf_proba)
xgb_auc = roc_auc_score(y_test, xgb_proba)

if xgb_auc >= rf_auc:
    best_model = xgb_model
    best_name  = 'XGBoost'
    best_auc   = xgb_auc
else:
    best_model = rf_model
    best_name  = 'RandomForest'
    best_auc   = rf_auc

print(f'🏆 Best model: {best_name} (ROC-AUC = {best_auc:.4f})')

## 💾 Step 10: Save Artifacts for FastAPI

In [ ]:
os.makedirs('model/artifacts', exist_ok=True)

# Save all artifacts
joblib.dump(best_model,    'model/artifacts/model.pkl')
joblib.dump(rf_model,      'model/artifacts/rf_model.pkl')
joblib.dump(xgb_model,     'model/artifacts/xgb_model.pkl')
joblib.dump(scaler,        'model/artifacts/scaler.pkl')
joblib.dump(le,            'model/artifacts/label_encoder.pkl')
joblib.dump(FEATURE_COLS,  'model/artifacts/feature_cols.pkl')

# Save model metadata
import json
metadata = {
    'best_model': best_name,
    'best_roc_auc': round(best_auc, 4),
    'rf_roc_auc': round(rf_auc, 4),
    'xgb_roc_auc': round(xgb_auc, 4),
    'feature_cols': FEATURE_COLS,
    'label_classes': list(le.classes_),
    'smote_used': True
}
with open('model/artifacts/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('✅ Saved artifacts:')
for f in os.listdir('model/artifacts'):
    size = os.path.getsize(f'model/artifacts/{f}')
    print(f'  📦 {f:35s} {size/1024:.1f} KB')

## 🧪 Step 11: Test Prediction Pipeline (same as FastAPI will use)

In [ ]:
def predict_failure(machine_type, air_temp, process_temp,
                    rotational_speed, torque, tool_wear,
                    model=best_model, scaler=scaler, le=le):
    """
    Replicates the FastAPI /predict endpoint logic.
    Returns: dict with prediction and probability.
    """
    # Encode type
    type_enc = le.transform([machine_type])[0]

    # Compute engineered features
    temp_delta              = process_temp - air_temp
    power_watts             = torque * (rotational_speed * 2 * np.pi / 60)
    wear_torque_interaction = tool_wear * torque
    speed_torque_ratio      = rotational_speed / (torque + 1e-6)
    wear_bin                = int(pd.cut([tool_wear], bins=[0, 60, 120, 180, 300], labels=[0, 1, 2, 3])[0])

    # Q90 of process temp from training data (precomputed)
    process_temp_q90 = df['Process temperature [K]'].quantile(0.90)
    speed_q10        = df['Rotational speed [rpm]'].quantile(0.10)
    high_temp_flag   = int(process_temp > process_temp_q90)
    low_speed_flag   = int(rotational_speed < speed_q10)

    features = np.array([[
        type_enc, air_temp, process_temp, rotational_speed,
        torque, tool_wear, temp_delta, power_watts,
        wear_torque_interaction, speed_torque_ratio,
        wear_bin, high_temp_flag, low_speed_flag
    ]])

    features_scaled = scaler.transform(features)
    prediction      = model.predict(features_scaled)[0]
    probability     = model.predict_proba(features_scaled)[0][1]

    risk = 'LOW' if probability < 0.2 else \
           'MEDIUM' if probability < 0.5 else \
           'HIGH' if probability < 0.75 else 'CRITICAL'

    return {
        'failure_predicted': bool(prediction),
        'failure_probability': round(float(probability), 4),
        'risk_level': risk,
        'message': '⚠️ Maintenance required!' if prediction else '✅ Machine operating normally.'
    }


# Test cases
print('--- Test 1: Normal operation ---')
print(predict_failure('M', 298.1, 308.6, 1551, 42.8, 10))

print('\n--- Test 2: High wear + high torque (likely failure) ---')
print(predict_failure('H', 302.0, 312.5, 1200, 68.0, 250))

print('\n--- Test 3: Low quality machine, high wear ---')
print(predict_failure('L', 300.5, 310.8, 1350, 55.0, 200))

## ✅ Summary

In [ ]:
print('='*55)
print('      AI4I 2020 PREDICTIVE MAINTENANCE — SUMMARY')
print('='*55)
print(f'  Dataset rows        : {len(df):,}')
print(f'  Failure rate        : {df["Machine failure"].mean()*100:.1f}%')
print(f'  Features used       : {len(FEATURE_COLS)} (incl. 7 engineered)')
print(f'  Imbalance handled   : SMOTE oversampling')
print(f'  Random Forest AUC   : {rf_auc:.4f}')
print(f'  XGBoost AUC         : {xgb_auc:.4f}')
print(f'  Best model          : {best_name}')
print(f'  Artifacts saved to  : model/artifacts/')
print('='*55)
print('  🚀 Ready to plug into FastAPI!')